<a href="https://colab.research.google.com/github/Maziger/Laksegate-master-thesis/blob/main/POC/TimeXer/run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

user = "Maziger"
repo = "Laksegate-master-thesis"

# remove local directory if it already exists
if os.path.isdir(repo):
    !rm -rf {repo}

!git clone https://github.com/{user}/{repo}.git
%cd Laksegate-master-thesis/

Cloning into 'Laksegate-master-thesis'...
remote: Enumerating objects: 320, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 320 (delta 4), reused 10 (delta 4), pack-reused 306 (from 2)
Receiving objects: 100% (320/320), 53.04 MiB | 20.23 MiB/s, done.
Resolving deltas: 100% (173/173), done.
/content/Laksegate-master-thesis


In [4]:
!ls

checkpoints    figures	  requirements.txt		 run.py
data_provider  layers	  result_long_term_forecast.txt  scripts
dataset        models	  results			 test_results
exp	       README.md  run.ipynb			 utils


In [3]:
%cd POC
%cd TimeXer

/content/Laksegate-master-thesis/POC
/content/Laksegate-master-thesis/POC/TimeXer


In [2]:
!pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [5]:
!pip install -q torch pandas scikit-learn matplotlib tqdm einops patool sktime reformer-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 16.8 MB/s eta 0:00:00


In [1]:
!python run.py --is_training 1 --task_name long_term_forecast \
  --root_path ./dataset/EPF/ --data_path DE.csv \
  --model_id DE_168_24 --model TimeXer --data custom \
  --features MS --seq_len 168 --pred_len 24 --e_layers 1 \
  --enc_in 3 --dec_in 3 --c_out 1 --des Timexer-MS \
  --patch_len 24 --batch_size 4 --d_model 512 --itr 1

python3: can't open file '/content/run.py': [Errno 2] No such file or directory


In [6]:
import sys, random
import numpy as np
import torch

sys.path.insert(0, '.')

fix_seed = 2021
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

import argparse
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

args = argparse.Namespace(
    task_name='long_term_forecast',
    is_training=1,
    model_id='DE_168_24',
    model='TimeXer',
    data='custom',
    root_path='./dataset/EPF/',
    data_path='DE.csv',
    features='MS',
    target='OT',
    freq='h',
    checkpoints='./checkpoints/',
    seq_len=168,
    label_len=48,
    pred_len=24,
    patch_len=24,
    e_layers=1,
    enc_in=3,
    dec_in=3,
    c_out=1,
    d_model=512,
    n_heads=8,
    d_layers=1,
    d_ff=2048,
    batch_size=4,
    train_epochs=10,
    patience=3,
    learning_rate=0.0001,
    des='Timexer-MS',
    loss='MSE',
    lradj='type1',
    itr=1,
    num_workers=2,
    use_amp=False,
    use_norm=1,
    inverse=False,
    embed='timeF',
    activation='gelu',
    factor=1,
    dropout=0.1,
    distil=True,
    output_attention=False,
    moving_avg=25,
    top_k=5,
    num_kernels=6,
    expand=2,
    d_conv=4,
    seg_len=48,
    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
    augmentation_ratio=0,
    use_dtw=False,
    seasonal_patterns='Monthly',
    mask_rate=0.25,
    anomaly_ratio=0.25,
    channel_independence=1,
    decomp_method='moving_avg',
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    use_gpu=True,
    gpu=0,
    use_multi_gpu=False,
    devices='0',
)

args.use_gpu = torch.cuda.is_available()
print(f"GPU available: {args.use_gpu}")
if args.use_gpu:
    print(torch.cuda.get_device_name(0))

exp = Exp_Long_Term_Forecast(args)

setting = (f"{args.task_name}_{args.model_id}_{args.model}_{args.data}_"
           f"ft{args.features}_sl{args.seq_len}_ll{args.label_len}_pl{args.pred_len}_"
           f"dm{args.d_model}_nh{args.n_heads}_el{args.e_layers}_dl{args.d_layers}_"
           f"df{args.d_ff}_expand{args.expand}_dc{args.d_conv}_fc{args.factor}_"
           f"eb{args.embed}_dt{args.distil}_{args.des}_0")

print(f'>>>>>>>start training: {setting}')
exp.train(setting)

print(f'>>>>>>>testing: {setting}')
exp.test(setting)
torch.cuda.empty_cache()


GPU available: True
Tesla T4
Use GPU: cuda:0
>>>>>>>start training: long_term_forecast_DE_168_24_TimeXer_custom_ftMS_sl168_ll48_pl24_dm512_nh8_el1_dl1_df2048_expand2_dc4_fc1_ebtimeF_dtTrue_Timexer-MS_0
train 36500
val 5219
test 10460
	iters: 100, epoch: 1 | loss: 1.5085258
	speed: 0.0531s/iter; left time: 4844.0731s
	iters: 200, epoch: 1 | loss: 0.3349008
	speed: 0.0119s/iter; left time: 1086.1041s
	iters: 300, epoch: 1 | loss: 1.3695457
	speed: 0.0122s/iter; left time: 1113.1136s
	iters: 400, epoch: 1 | loss: 0.3256595
	speed: 0.0151s/iter; left time: 1371.1944s
	iters: 500, epoch: 1 | loss: 0.6055267
	speed: 0.0140s/iter; left time: 1270.2531s
	iters: 600, epoch: 1 | loss: 0.4027645
	speed: 0.0113s/iter; left time: 1021.9189s
	iters: 700, epoch: 1 | loss: 0.6246306
	speed: 0.0111s/iter; left time: 1003.3954s
	iters: 800, epoch: 1 | loss: 0.2040088
	speed: 0.0112s/iter; left time: 1015.6488s
	iters: 900, epoch: 1 | loss: 0.6921748
	speed: 0.0135s/iter; left time: 1219.6947s
	iters: 10